# 03. Testo vertinimas ir hipotezės H1–H3

> **Peržiūros notebook.** Šaltiniai: `reports/tables/main_results.csv`, `hypotheses.csv`, `h3_nested.csv`; grafikai `../reports/figures/`.


## Kas čia vyksta paprastai

Kai modeliai ir slenksčiai užšaldyti, testas **atrakinamas lygiai vieną kartą**. Skaičiuojamas jautrumas (kiek piktybinių pagauta), specifiškumas (kiek gerybinių nepalietė aliarmas), AUC, Brier ir kaina EC. Hipotezės H1/H2 tikrinamos šiame 113 atvejų teste; H3 — atskiru įdėtiniu CV **po** testo ir H1 neperrašo.

Jei kas nors po pirmos lentelės „pataisytų“ slenkstį pagal testą — tai jau būtų žvilgsnis į atsakymus. Todėl antras `evaluate_test` kvietimas meta `WDBC:Eval:AlreadyUnlocked`.


## `evalx/evaluate_test.m` ir `metrics.m`

```matlab
% evalx/evaluate_test.m (fragmentas)
lockPath = fullfile(cfg.paths.reports, 'test_unlocked.flag');
if exist(lockPath, 'file')
    error('WDBC:Eval:AlreadyUnlocked', ...
        'evaluate_test: testas jau atrakintas. Antras kvietimas draudziamas.');
end
```

Kolokviumo (7): Se = TP/(TP+FN), Sp = TN/(TN+FP). Wilson 95 % PI Se/Sp. AUC — Mann–Whitney / `perfcurve` (8); Brier (9): $\mathrm{BS}=\frac{1}{n}\sum_i (p_i-y_i)^2$.

```matlab
% evalx/metrics.m (fragmentas)
yhat = p >= t;
m.Se = m.TP / max(m.TP + m.FN, eps);
m.Sp = m.TN / max(m.TN + m.FP, eps);
[m.Se_lo, m.Se_hi] = wilson_interval(m.TP, nPos);
```


## Pagrindinė testo lentelė — taškas `t_se98`

Iš `main_results.csv` eilučių `point=t_se98`. Apvalinta ataskaitai; tikslūs laukai CSV.

| modelis | t | Se | Sp | FN | FP | AUC | Brier | EC |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| majority | 0,50 | 0,000 | 1,000 | 42 | 0 | 0,5000 | 0,3717 | 210 |
| logreg | 0,21 | 1,000 | 0,5634 | 0 | 31 | 0,9883 | 0,0721 | 31 |
| svm_rbf | 0,34 | 0,9524 | 0,9296 | 2 | 5 | 0,9863 | 0,0430 | 15 |
| mlp | 0,44 | 0,9286 | 0,9859 | 3 | 1 | 0,9933 | 0,0328 | 16 |
| rbfnet | 0,22 | 0,9524 | 0,8873 | 2 | 8 | 0,9755 | 0,0525 | 18 |
| svm_linear | 0,16 | 0,9762 | 0,8873 | 1 | 8 | 0,9953 | 0,0312 | 13 |
| svm_mean | 0,08 | 0,9762 | 0,7183 | 1 | 20 | 0,9745 | 0,0619 | 25 |
| svm_worst | 0,09 | 1,000 | 0,8732 | 0 | 9 | 0,9963 | 0,0249 | 9 |

Skaitymas: LR `t_se98` pagavo visus 42 M, bet 31 FP (Sp ≈ 0,56). SVM-RBF praleido 2 M, bet FP tik 5 (Sp ≈ 0,93). AUC visų (išskyrus B0) ≈ 0,97–1,00 — **lubų efektas** rangui, ne specifiškumui vs LR.

`main_results.csv` **neturi** atskirų AUC/Brier pasikliautinųjų intervalų pagal modelį — toje vietoje ataskaitoje TRŪKSTA DUOMENŲ, čia neįsivaizduojama.


## ROC kreivės (testas)

![ROC kreivės teste](../reports/figures/roc.png)

**Parašas.** Testo ROC: x ašis 1−Sp, y ašis Se. Kreivės B1/M1/M2/M3 ir abliacijos SVM beveik visos aukštai kairėje (AUC ≥ 0,975, `main_results.csv`). Skirtumas, kuris rūpi H1, **nėra** plotas po kreive, o **operacinis taškas** `t_se98` (Se–Sp pora). B0 — įstrižainė AUC = 0,5.


## Kalibracija

![Patikimumo diagrama](../reports/figures/calibration.png)

**Parašas.** Patikimumo (reliability) diagrama teste: x — vidutinė prognozuota P(M) dėžutėje, y — faktinė dalis M. Idealu — įstrižainė. SVM-RBF Brier = 0,0430, LR = 0,0721 (`main_results.csv`). H2 tikrina, ar SVM Brier nėra blogesnis už LR daugiau kaip 0,01 ir ar kalibracijos nuolydis protingas.


## SVM-RBF painiavos matrica (`t_se98`)

![SVM-RBF painiavos matrica](../reports/figures/confusion_svm_rbf.png)

**Parašas.** SVM-RBF teste ties `t=0,34`: TP = 40, FN = 2, FP = 5, TN = 66 (`main_results.csv`, `svm_rbf` / `t_se98`). Du praleisti piktybiniai ir penki klaidingi aliarmos — klaidos analizė 04 notebook'e.


## Tikėtinos kainos kreivė

![EC ir Se vs slenkstis](../reports/figures/cost_curve.png)

**Parašas.** `cost_curve.png`: EC(t) ir/ar Se vs t. `t_cost=0,33` ir `t_se98=0,34` SVM-RBF duoda **tą pačią** testo EC = 15 (2×5 + 5×1). A7 abliacija (04) rodo, kad 1:1–10:1 kainų santykiai šioje imtyje **nekeičia** FN/FP ties `t_cost`.


## Hipotezės — `hypotheses.csv`

Preregistracija (`reports/preregistration.md`), **prieš** testą:

- **H1 (konjunkcija):** ties kiekvieno modelio `t_se98`: Sp_SVM − Sp_LR ≥ 0,03 **ir** bootstrap 2000 95 % PI apatinė riba > 0; **ir** DeLong AUC PI apatinė ≥ −0,005.
- **H2:** SVM Brier teste ne blogesnis už LR daugiau kaip 0,01; kalibracijos nuolydis kerta [0,8; 1,25].
- **H3:** 10×5 įdėtiniame CV ΔSp ties Se ≥ 0,98 teigiamas ≥ 70 % skaidinių; **neatstoja H1**.

| hipotezė | taškas | PI apačia | PI viršus | accepted |
|---|---:|---:|---:|---:|
| H1_dSp | 0,3662 | 0,2568 | 0,4786 | 1 |
| H1_AUC | −0,0020 | −0,0074 | 0,0034 | 0 |
| **H1** | 0 | — | — | **0** |
| H2_BS | −0,0291 | −0,0496 | −0,0078 | 1 |
| H2_slope | 1,0973 | 0,7893 | 2,7525 | 1 |
| **H2** | 1 | — | — | **1** |
| H3_frac | 1 | — | — | 1 |
| **H3** | 1 | — | — | **1** |

**H1 ATMESTA.** ΔSp ≈ 0,37 su PI toli virš 0 — specifiškumo dalis priimta. AUC ne-prastesnumas **nepraeina**: `ci_lo = −0,0074 < −0,005`. Tai validus rezultatas, ne „nepavykęs projektas“. Prototipo taisyklė po H1 atmetimo — LR (egzamino promptas) arba tiesinis SVM, jei A2 geresnis (plano 4.1).

**H2 PRIIMTA.** SVM Brier mažesnis (`H2_BS` taškas −0,029); nuolydžio PI kerta [0,8; 1,25] (intervalas platus dėl n = 113).

**H3 PRIIMTA.** `h3_nested.csv`: visos 50 išorinių foldų ΔSp > 0, `frac_positive = 1` ≥ 0,70. H1 eilutė `hypotheses.csv` **neperrašyta**.


## Ką daryti su H1 atmetimu

AUC lubos: LR jau 0,988, SVM 0,986 — DeLong skirtumas ~0 su PI, kuris kerta −0,005. Specifiškumas vs LR **nėra** lubose: +0,37. Prototipas todėl negali pretenduoti į „SVM įrodytas geresnis pagal visą H1“.

Toliau: [04_abliacija_ir_klaidu_analize.ipynb](04_abliacija_ir_klaidu_analize.ipynb).
